# Assignment 1: Market Clearing (System Perspective)

### Import external libraries

In [1]:
import time
#for the calculation of the runtime of the code
start_time = time.time()

import gurobipy as gp
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from gurobipy import GRB

import network
from network import initialize_network, system_demand, LINES


## Step 0: Build a Relevant Case Study

Please select an electric power network from the following options:
1. IEEE 24-bus reliability test system: link.
2. IEEE reliability test system (2019 update): link.
3. IEEE power systems test cases (various cases with 14, 30, 57, 118, and 300 buses): link.
4. Case studies available in the open-source Julia platform PowerModels.jl: link.

You are also free to choose another case study. If some data is missing, please select reason-
able arbitrary values. For technical details on conventional generators and transmission lines, this link may be helpful (it corresponds to the IEEE 24-bus case study, but similar data can be used for other cases).

### IEEE 24-bus reliability test system

In [2]:
consumers, generators = initialize_network()
print(generators)
print(consumers)
print("Network initialized successfully.")

[Generator(unit=1, node=1), Generator(unit=2, node=2), Generator(unit=3, node=7), Generator(unit=4, node=13), Generator(unit=5, node=15), Generator(unit=6, node=15), Generator(unit=7, node=16), Generator(unit=8, node=18), Generator(unit=9, node=21), Generator(unit=10, node=22), Generator(unit=11, node=23), Generator(unit=12, node=23), Generator(unit=13, node=3), Generator(unit=14, node=5), Generator(unit=15, node=7), Generator(unit=16, node=16), Generator(unit=17, node=21), Generator(unit=18, node=23)]
[Consumer(load=1, node=1, share=0.038), Consumer(load=2, node=2, share=0.034), Consumer(load=3, node=3, share=0.063), Consumer(load=4, node=4, share=0.026), Consumer(load=5, node=5, share=0.025), Consumer(load=6, node=6, share=0.048), Consumer(load=7, node=7, share=0.044), Consumer(load=8, node=8, share=0.06), Consumer(load=9, node=9, share=0.061), Consumer(load=10, node=10, share=0.068), Consumer(load=11, node=13, share=0.093), Consumer(load=12, node=14, share=0.068), Consumer(load=13, 

### Additional Assumptions


*   Assume that the price bids of all producers are non-negative and equal to their marginal production cost. In particular, the production cost of renewable units is assumed to be zero. Additionally, these units offer their forecasted capacity, meaning their offer quantities vary over time.

*   For the bid price of price-elastic demands, use comparatively high values (relative to the generation cost of conventional units) to ensure that most demands are supplied. For inspiration, check the real bid price data in Nord Pool [link].
*   A potential source for wind power forecast data is available at this link (you may nor- malize the data to fit your case study). Another potential source for the renewable power generation data is renewables.ninja.
*   For transmission lines, you may assume a uniform reactance for all lines (e.g., 0.002 p.u., leading to a susceptance of 500 p.u.).

### The market-clearing price under a uniform pricing scheme.

In [3]:
# define the demand time at 16:00
total_consumption_16 = system_demand[16]
print(f"Total consumption at 16:00 is {total_consumption_16} MW")
#check if all generators are there
id_Gnerators = []
for i in generators:
    id_Gnerators.append(i.unit_id)
print(id_Gnerators)

Total consumption at 16:00 is 2464.965 MW
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


Define:
*   Generator production cost
*   Generator production capacity
*   Demand bid prize offer
*   Demand consumption

In [4]:
# list for the variable generators costs for each $/MWh
generator_cost = []
#list for the variable generators capacity in MWh
generator_capacity = []
for i in generators:
    generator_cost.append(i.cost_energy)
    generator_capacity.append(i.p_max)

#list with the variable demand bids prize
demand_bids = []
#list with consumption for each consumer
consumption = []
for i in consumers:
    demand_bids.append(i.price)
    consumption.append(i.share * total_consumption_16)

if sum(consumption) == total_consumption_16:
    print("Aggregation correct")

Aggregation correct


## Step 6: Reserve Market

In Lecture 6, you learn about the reserve market. We again use the model in Step 1. Assume the
TSO has determined that the upward reserve services to be procured in the reserve market is
15% of the total demand in the corresponding hour. This value is 10% for the downward reserve
services. Consider the same subset of conventional generators in Step 5 as flexible resources.
Please clear the reserve and day-ahead markets sequentially, following the current practice in
European electricity markets, and report the market-clearing prices for reserve and electricity.\\

How does the reserve market change prices in the day-ahead market?

### 6.1 Clear the reserve market


In [5]:
upward_reserve_services = 0.15  * total_consumption_16# 15% of the total demand as upward reserve requirement
downward_reserve_services = 0.10 * total_consumption_16 # 10% of the total demand as downward reserve requirement

#upward and downward reserve generating cost per Generators
#those are capacities not costs
upward_reserve_costs = []
downward_reserve_costs = []

# upward and downward genratating capacities for each generator
upward_gernating_capacities = []
downward_gernating_capacities = []
for i in range(12):
    if i in [7, 8, 9]:
        continue
    upward_reserve_costs.append(generators[i].cost_up_reserve)
    downward_reserve_costs.append(generators[i].cost_down_reserve)
    upward_gernating_capacities.append(generators[i].r_up)
    downward_gernating_capacities.append(generators[i].r_down)

print(upward_gernating_capacities)
print(downward_gernating_capacities)

selected_generator_capacities = [generator_capacity[i] for i in range(12) if i not in (7, 8, 9)]


[40, 40, 70, 180, 60, 30, 30, 60, 40]
[40, 40, 70, 180, 60, 30, 30, 60, 40]


In [6]:
#create the optimization model for the reseve market
model_reserve = gp.Model("Economic_Dispatch_Model_for_Reserve_Market")

Set parameter Username
Set parameter LicenseID to value 2773996
Academic license - for non-commercial use only - expires 2027-02-02


In [7]:
upward_reserve_capacity = [
    model_reserve.addVar(lb=0, ub = float('inf') , vtype=GRB.CONTINUOUS, name=f"urc_{i}")
    for i,v in enumerate(upward_reserve_costs)
]

downward_reserve_capacity = [
    model_reserve.addVar(lb=0, ub = float('inf') , vtype=GRB.CONTINUOUS, name=f"drc_{i}")
    for i,v in enumerate(downward_reserve_costs)
]

In [8]:
balance_constraint_upward = model_reserve.addConstr(gp.quicksum(upward_reserve_capacity[i] for i,v in enumerate(upward_reserve_costs)) == upward_reserve_services, name="balance_constraint_upward")

balance_constraint_downward = model_reserve.addConstr(gp.quicksum(downward_reserve_capacity[i] for i,v in enumerate(downward_reserve_costs)) == downward_reserve_services, name="balance_constraint_downward")

upward_capacity_constraints = [model_reserve.addConstr(upward_reserve_capacity[i] <= upward_gernating_capacities[i] , name=f"upward_capacity_constraint_{i}") for i in range(len(upward_reserve_costs))]

downward_capacity_constraints = [model_reserve.addConstr(downward_reserve_capacity[i] <= downward_gernating_capacities[i] , name=f"downward_capacity_constraint_{i}") for i in range(len(downward_reserve_costs))]

reserve_feasibility_constraints = [
    model_reserve.addConstr(
        upward_reserve_capacity[i] + downward_reserve_capacity[i] <= selected_generator_capacities[i],
        name=f"reserve_feasibility_{i}"
    )
    for i in range(len(selected_generator_capacities))
]

In [9]:
model_reserve.setObjective(
    gp.quicksum(downward_reserve_capacity[j] * downward_reserve_costs[j] for j in range(len(downward_reserve_costs)))
    + gp.quicksum(upward_reserve_capacity[i] * upward_reserve_costs[i] for i in range(len(upward_reserve_costs))),
    GRB.MINIMIZE
)
#model.setObjective(
    #gp.quicksum(generator_cost[i] * production_variables[i] for i in range(len(production_variables))),
    #GRB.MINIMIZE
#)

In [10]:
model_reserve.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[rosetta2] - Darwin 25.3.0 25D2128)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 29 rows, 18 columns and 54 nonzeros (Min)
Model fingerprint: 0x378e62fe
Model has 18 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 6e+02]

Presolve removed 26 rows and 13 columns
Presolve time: 0.00s
Presolved: 3 rows, 5 columns, 7 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.5966518e+03   2.243806e+01   0.000000e+00      0s
       2    5.3146398e+03   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.314639750e+03


In [11]:
#check status and print results 
if model_reserve.status == GRB.OPTIMAL:
    optimal_objective = model_reserve.objVal
    optimal_upward_production = [upward_reserve_capacity[i].x for i in range(len(upward_gernating_capacities))]
    optimal_downward_production = [downward_reserve_capacity[i].x for i in range(len(downward_gernating_capacities))]
    
    balance_dual_downward = balance_constraint_downward.Pi
    balance_dual_upward = balance_constraint_upward.Pi
    # use the existing (typo) capacaity_constraints variable
    upward_capacity_constraints_duals = [upward_capacity_constraints[i].Pi for i in range(len(upward_capacity_constraints))]
    downward_capacity_constraints_duals = [downward_capacity_constraints[i].Pi for i in range(len(downward_capacity_constraints))]
    print (f"Optimal objective value: {optimal_objective}")

    for index, optimal in enumerate(optimal_upward_production):
        print(f"Optimal upward production for Generator {id_Gnerators[index]:.2f}: {optimal} MW")

    for index, optimal in enumerate(optimal_downward_production):
        print(f"Optimal downward production for Generator {id_Gnerators[index]:.2f}: {optimal} MW")

    

    print(f"Dual variable for upward balance constraint: {balance_dual_upward:.2f}")
    print(f"Dual variable for downward balance constraint: {balance_dual_downward:.2f}")

    #for index, dual in enumerate(upward_capacity_constraints_duals):
        #print(f"Dual variable for upward capacity constraint of Generator {id_Gnerators[index]}: {dual}")

    #for index, dual in enumerate(downward_capacity_constraints_duals):
        #print(f"Dual variable for downward capacity constraint of Generator {id_Gnerators[index]}: {dual}")

else:
    print(f"optimization of model {model.ModelName} was not successful. Status code: {model.status}")

Optimal objective value: 5314.63975
Optimal upward production for Generator 1.00: 19.74475000000001 MW
Optimal upward production for Generator 2.00: 40.0 MW
Optimal upward production for Generator 3.00: 70.0 MW
Optimal upward production for Generator 4.00: 180.0 MW
Optimal upward production for Generator 5.00: 60.0 MW
Optimal upward production for Generator 6.00: 0.0 MW
Optimal upward production for Generator 7.00: 0.0 MW
Optimal upward production for Generator 8.00: 0.0 MW
Optimal upward production for Generator 9.00: 0.0 MW
Optimal downward production for Generator 1.00: 0.0 MW
Optimal downward production for Generator 2.00: 0.0 MW
Optimal downward production for Generator 3.00: 66.49650000000003 MW
Optimal downward production for Generator 4.00: 180.0 MW
Optimal downward production for Generator 5.00: 0.0 MW
Optimal downward production for Generator 6.00: 0.0 MW
Optimal downward production for Generator 7.00: 0.0 MW
Optimal downward production for Generator 8.00: 0.0 MW
Optimal down

### Clear day-ahead market according to step 1 with inluence of reserve market


In [12]:
optimal_upward_production = (
    optimal_upward_production[:7] +
    [0, 0, 0] +
    optimal_upward_production[7:] +
    [0, 0, 0, 0, 0,0] 
)

optimal_downward_production = (
    optimal_downward_production[:7] +
    [0, 0, 0] +
    optimal_downward_production[7:] +
    [0, 0, 0, 0, 0,0] 
)

print(optimal_downward_production)

generator_capacity_v2 = [generator_capacity[i] - optimal_upward_production[i] for i in range(len(generator_capacity))]
print(generator_capacity)
print(optimal_upward_production)
print(generator_capacity_v2)

[0.0, 0.0, 66.49650000000003, 180.0, 0.0, 0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0, 0, 0, 0, 0, 0]
[152, 152, 350, 591, 60, 155, 155, 400, 400, 300, 310, 350, 200, 200, 200, 200, 200, 200]
[19.74475000000001, 40.0, 70.0, 180.0, 60.0, 0.0, 0.0, 0, 0, 0, 0.0, 0.0, 0, 0, 0, 0, 0, 0]
[132.25525, 112.0, 280.0, 411.0, 0.0, 155.0, 155.0, 400, 400, 300, 310.0, 350.0, 200, 200, 200, 200, 200, 200]


In [13]:
#create the optimization model
model_day_ahead_ex6 = gp.Model("Economic_Dispatch_Model_ex6")

In [14]:
production_variables_ex6 = [
    model_day_ahead_ex6.addVar(lb = optimal_downward_production[i], ub = float('inf') , vtype=GRB.CONTINUOUS, name=f"p_{i}")
    for i,v in enumerate(generator_capacity)
]

consumption_variables_ex6 = [
    model_day_ahead_ex6.addVar(lb=0, ub = v , vtype=GRB.CONTINUOUS, name=f"p_{i}")
    for i,v in enumerate(consumption)
]

In [15]:
balance_constraint_ex6 = model_day_ahead_ex6.addConstr(gp.quicksum(consumption_variables_ex6[i] for i,v in enumerate(consumption)) - gp.quicksum(production_variables_ex6[i] for i,v in enumerate(generator_capacity)) == 0, name="balance_constraint")

capacaity_constraints_ex6 = [model_day_ahead_ex6.addConstr(production_variables_ex6[i] <= generator_capacity_v2[i], name=f"capacity_constraint_{i}") for i in range(len(generator_capacity))]

In [16]:
model_day_ahead_ex6.setObjective(
    gp.quicksum(consumption_variables_ex6[j] * demand_bids[j] for j in range(len(consumption_variables_ex6)))
    - gp.quicksum(generator_cost[i] * production_variables_ex6[i] for i in range(len(production_variables_ex6))),
    GRB.MAXIMIZE
)

In [17]:
model_day_ahead_ex6.optimize()
if model_day_ahead_ex6.status == GRB.INFEASIBLE:
    model_day_ahead_ex6.computeIIS()

    print("\nConstraints in the IIS:")
    for c in model_day_ahead_ex6.getConstrs():
        if c.IISConstr:
            print(f"- {c.ConstrName}")

    print("\nVariable bounds in the IIS:")
    for v in model_day_ahead_ex6.getVars():
        if v.IISLB:
            print(f"- Lower bound of {v.VarName}: {v.LB}")
        if v.IISUB:
            print(f"- Upper bound of {v.VarName}: {v.UB}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[rosetta2] - Darwin 25.3.0 25D2128)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 19 rows, 35 columns and 53 nonzeros (Max)
Model fingerprint: 0x6ab41f86
Model has 28 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+02]
  Bounds range     [6e+01, 3e+02]
  RHS range        [1e+02, 4e+02]

Presolve removed 18 rows and 10 columns
Presolve time: 0.00s
Presolved: 1 rows, 25 columns, 25 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.5548906e+05   8.980856e+01   0.000000e+00      0s
       1    4.5138388e+05   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.513838816e+05


In [18]:
#check status and print results 
if model_day_ahead_ex6.status == GRB.OPTIMAL:
    optimal_objective_ex6 = model_day_ahead_ex6.objVal
    optimal_production_variables_ex6 = [production_variables_ex6[i].x for i in range(len(generators))]
    optimal_consumption_variables_ex6 = [consumption_variables_ex6[i].x for i in range(len(consumers))]
    balance_dual_ex6 = balance_constraint_ex6.Pi
    # use the existing (typo) capacaity_constraints variable
    capacity_duals_ex6 = [capacaity_constraints_ex6[i].Pi for i in range(len(generators))]

    print (f"Optimal objective value: {optimal_objective_ex6}")

    for index, optimal in enumerate(optimal_production_variables_ex6):
        print(f"Optimal production for Generator {id_Gnerators[index]:.2f}: {optimal} MW")

    for index, optimal in enumerate(optimal_consumption_variables_ex6):
        print(f"Optimal demand for Consumer {index}: {optimal:.2f} MW")

    

    for index, dual in enumerate(capacity_duals_ex6):
        print(f"Dual variable for capacity constraint of Generator {id_Gnerators[index]}: {dual}")

    print(f"Dual variable for balance constraint: {balance_dual_ex6:.2f}")

else:
    print(f"optimization of model {model_day_ahead_ex6.ModelName} was not successful. Status code: {model_day_ahead_ex6.status}")

Optimal objective value: 451383.88156000007
Optimal production for Generator 1.00: 0.0 MW
Optimal production for Generator 2.00: 0.0 MW
Optimal production for Generator 3.00: 66.49650000000003 MW
Optimal production for Generator 4.00: 180.0 MW
Optimal production for Generator 5.00: 0.0 MW
Optimal production for Generator 6.00: 0.0 MW
Optimal production for Generator 7.00: 0.0 MW
Optimal production for Generator 8.00: 318.46849999999995 MW
Optimal production for Generator 9.00: 400.0 MW
Optimal production for Generator 10.00: 300.0 MW
Optimal production for Generator 11.00: 0.0 MW
Optimal production for Generator 12.00: 0.0 MW
Optimal production for Generator 13.00: 200.0 MW
Optimal production for Generator 14.00: 200.0 MW
Optimal production for Generator 15.00: 200.0 MW
Optimal production for Generator 16.00: 200.0 MW
Optimal production for Generator 17.00: 200.0 MW
Optimal production for Generator 18.00: 200.0 MW
Optimal demand for Consumer 0: 93.67 MW
Optimal demand for Consumer 1: 8

In [19]:
status = model_day_ahead_ex6.status

if status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model_day_ahead_ex6.objVal}")

elif status == GRB.INFEASIBLE:
    print("Model is infeasible: no solution satisfies all constraints.")

elif status == GRB.UNBOUNDED:
    print("Model is unbounded: the objective can improve without limit.")

elif status == GRB.INF_OR_UNBD:
    print("Model is infeasible or unbounded.")

elif status == GRB.TIME_LIMIT:
    print("Time limit reached.")
    if model_day_ahead_ex6.SolCount > 0:
        print(f"Best solution found: {model_day_ahead_ex6.objVal}")
    else:
        print("No feasible solution found before time limit.")

else:
    print(f"Optimization stopped with status code {status}")

Optimal objective value: 451383.88156000007


In [20]:
end_time = time.time()
runtime = end_time - start_time

# Format into hours, minutes, seconds
hours = int(runtime // 3600)
minutes = int((runtime % 3600) // 60)
seconds = runtime % 60

print("\n" + "="*40)
print("⏱️  Notebook Runtime")
print("="*40)
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))}")
print(f"End time:   {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time))}")
print(f"Duration:   {hours}h {minutes}min {seconds:.2f}s")
print("="*40)


⏱️  Notebook Runtime
Start time: 2026-03-23 09:47:21
End time:   2026-03-23 09:47:22
Duration:   0h 0min 1.06s
